# 07 · Localization validation (mean IoU)
**Paper:** Step 8 · Section III-A

We measure how closely the detector's *Pathology Tissue* boxes match the manual
annotations on the **150-image validation split**:

* predicted boxes (confidence ≥ 0.5) and ground-truth boxes are **greedily matched in
  descending IoU**, and each box is used at most once;
* **mean IoU** is the average over matched pairs;
* **strict mean IoU** also counts each unmatched ground-truth box as IoU = 0, which penalises
  missed tissue.

**Output** → `outputs/val_iou_per_image.csv`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import pandas as pd
from ultralytics import YOLO

from requilt.boxes import greedy_match
from requilt.data import label_path_for, read_yolo_labels, yolo_to_xyxy
from requilt.recovery import tissue_candidates

R = cfg.recovery
VAL_IMAGES = P.output_dir / "yolo_dataset" / "images" / "val"
VAL_LABELS = P.output_dir / "yolo_dataset" / "labels" / "val"

## 1 · Predict on the validation images

In [ ]:
model = YOLO(P.weights)
TISSUE = {name: idx for idx, name in model.names.items()}[R.target_class]

results = model.predict(source=str(VAL_IMAGES), imgsz=cfg.train.img_size,
                        conf=R.predict_conf, stream=True, verbose=False)   # lazy, low memory

## 2 · Match predictions to ground truth

In [ ]:
rows, all_ious, n_missed, n_extra = [], [], 0, 0

for r in results:
    name = Path(r.path).name
    img_h, img_w = r.orig_shape

    gt = [yolo_to_xyxy(*box, img_w, img_h)
          for cls_id, *box in read_yolo_labels(label_path_for(name, VAL_LABELS)) if cls_id == TISSUE]
    pred = [xyxy for xyxy, _, _ in tissue_candidates(r, TISSUE, R.min_confidence)]

    ious, missed, extra = greedy_match(gt, pred)
    all_ious += ious
    n_missed += missed
    n_extra += extra
    rows.append({
        "filename": name,
        "n_gt_boxes": len(gt),
        "n_pred_boxes": len(pred),
        "n_matched": len(ious),
        "mean_iou_this_image": sum(ious) / len(ious) if ious else None,
        "n_missed_gt": missed,
        "n_false_positive_pred": extra,
    })

pd.DataFrame(rows).to_csv(P.output_dir / "val_iou_per_image.csv", index=False)

## 3 · Summary
Expected: 180 ground-truth boxes · 165 matched · 15 missed · 6 unmatched predictions ·
**mean IoU 0.899** · **strict mean IoU 0.824**.

In [ ]:
mean_iou = sum(all_ious) / max(len(all_ious), 1)
strict_iou = sum(all_ious) / max(len(all_ious) + n_missed, 1)

print(f"ground-truth tissue boxes : {len(all_ious) + n_missed}")
print(f"matched                   : {len(all_ious)}")
print(f"missed                    : {n_missed}")
print(f"unmatched predictions     : {n_extra}")
print(f"mean IoU (matched)        : {mean_iou:.3f}")
print(f"strict mean IoU           : {strict_iou:.3f}")